# Semantic Router Config Generation - Model Training

This notebook fine-tunes a base LLM (Llama 3.1 8B or Qwen 2.5 7B) to generate semantic-router YAML configurations from natural language descriptions.

## Prerequisites

Before running this notebook, ensure you have:
1. ✅ Downloaded base model locally and uploaded `base_model.zip` to Google Drive
2. ✅ Uploaded `train_config_gen_lora.py` to Google Drive
3. ✅ Uploaded `train.jsonl` and `val.jsonl` to Google Drive
4. ✅ (If using Llama) HuggingFace access token ready

See `PRE_COLAB_CHECKLIST.md` for detailed preparation steps.


## Step 1: Install Dependencies


In [ ]:
# Install required packages
!pip install -q transformers datasets peft accelerate torch pyyaml tqdm huggingface_hub

print("✅ Dependencies installed")


## Step 2: Connect to Google Drive

Connect to Google Drive to access uploaded files (base model, training script, dataset).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Your Drive folder path
DRIVE_FOLDER = "/content/drive/MyDrive/vllm/semantic-router/sr-config-gen"

print(f"✅ Connected to Google Drive")
print(f"📁 Using Drive folder: {DRIVE_FOLDER}")


## Step 3: Load Files from Drive

Load the base model, training script, and dataset files from Google Drive.


In [ ]:
import os
import shutil
import tarfile

# Clean up any existing files
!rm -rf base_model base_model.zip base_model.tar.gz train_config_gen_lora.py dataset

# Copy files from Drive
print("📥 Loading files from Google Drive...")
print(f"📁 Looking in: {DRIVE_FOLDER}")
print("")

# Base model - try multiple locations and formats
model_loaded = False
model_paths = [
    f"{DRIVE_FOLDER}/base_model.tar.gz",
    f"{DRIVE_FOLDER}/training/base_model.tar.gz",
    f"{DRIVE_FOLDER}/base_model.zip",
    f"{DRIVE_FOLDER}/training/base_model.zip",
]

for model_path in model_paths:
    if os.path.exists(model_path):
        print(f"✅ Found model at: {model_path}")
        if model_path.endswith('.tar.gz'):
            !cp "{model_path}" .
            !tar -xzf base_model.tar.gz
            print("✅ Base model extracted from tar.gz")
        else:
            !cp "{model_path}" .
            !unzip -q -o base_model.zip
            print("✅ Base model extracted from zip")
        model_loaded = True
        break

if not model_loaded:
    print("⚠️  base_model.tar.gz or base_model.zip not found!")
    print("   Searched in:")
    for p in model_paths:
        print(f"   - {p}")

# Training script - try multiple locations
script_loaded = False
script_paths = [
    f"{DRIVE_FOLDER}/train_config_gen_lora.py",
    f"{DRIVE_FOLDER}/training/train_config_gen_lora.py",
]

for script_path in script_paths:
    if os.path.exists(script_path):
        !cp "{script_path}" .
        print("✅ Training script loaded")
        script_loaded = True
        break

if not script_loaded:
    print("⚠️  train_config_gen_lora.py not found!")
    print("   Searched in:")
    for p in script_paths:
        print(f"   - {p}")

# Dataset - try multiple locations
os.makedirs("dataset/processed", exist_ok=True)
train_loaded = False
val_loaded = False
test_loaded = False

train_paths = [
    f"{DRIVE_FOLDER}/train.jsonl",
    f"{DRIVE_FOLDER}/dataset/processed/train.jsonl",
    f"{DRIVE_FOLDER}/training/dataset/processed/train.jsonl",
]

val_paths = [
    f"{DRIVE_FOLDER}/val.jsonl",
    f"{DRIVE_FOLDER}/dataset/processed/val.jsonl",
    f"{DRIVE_FOLDER}/training/dataset/processed/val.jsonl",
]

test_paths = [
    f"{DRIVE_FOLDER}/test.jsonl",
    f"{DRIVE_FOLDER}/dataset/processed/test.jsonl",
    f"{DRIVE_FOLDER}/training/dataset/processed/test.jsonl",
]

for train_path in train_paths:
    if os.path.exists(train_path):
        !cp "{train_path}" dataset/processed/
        print("✅ Training dataset loaded")
        train_loaded = True
        break

for val_path in val_paths:
    if os.path.exists(val_path):
        !cp "{val_path}" dataset/processed/
        print("✅ Validation dataset loaded")
        val_loaded = True
        break

for test_path in test_paths:
    if os.path.exists(test_path):
        !cp "{test_path}" dataset/processed/
        print("✅ Test dataset loaded")
        test_loaded = True
        break

if not train_loaded:
    print("⚠️  train.jsonl not found!")
if not val_loaded:
    print("⚠️  val.jsonl not found!")
if not test_loaded:
    print("⚠️  test.jsonl not found! ← REQUIRED FOR EVALUATION")
    print("   Upload test.jsonl to your Drive folder")

# Verify files
print("\n📊 Files loaded:")
!ls -lh base_model/ 2>/dev/null | head -5 || echo "Base model not found"
!ls -lh train_config_gen_lora.py 2>/dev/null || echo "Training script not found"
!ls -lh dataset/processed/*.jsonl 2>/dev/null || echo "Dataset files not found"


## Step 4: Configure Training

Set your training parameters. Adjust based on your GPU memory and needs.


In [ ]:
# Training configuration
BASE_MODEL = "./base_model"  # Use local model from Drive
# Alternative: Use HuggingFace model directly (if you have access)
# BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct"

TRAIN_FILE = "dataset/processed/train.jsonl"
VAL_FILE = "dataset/processed/val.jsonl"
OUTPUT_DIR = "models/config_gen_v1.0.0"

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 4  # Reduce to 2 if OOM
LEARNING_RATE = 2e-5
LORA_RANK = 16
LORA_ALPHA = 32
MAX_LENGTH = 4096  # Reduce to 2048 if OOM

print("📋 Training Configuration:")
print(f"  Base model: {BASE_MODEL}")
print(f"  Train file: {TRAIN_FILE}")
print(f"  Val file: {VAL_FILE}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")


## Step 5: Evaluate Base Model (Baseline)

Evaluate the base model before training to establish a baseline. This will help you understand if training improved the model.

⚠️ **IMPORTANT**: Let this complete! Don't interrupt (^C) - it needs to finish to show results.


In [ ]:
# Evaluate base model (baseline)
print("📊 Evaluating Base Model (Baseline)...")
print("="*70)
print("⚠️  IMPORTANT: Let this complete to see the results!")
print("   This will take a few minutes. Don't interrupt (^C)!")
print("")

!python train_config_gen_lora.py \
    --mode evaluate \
    --base-model "./base_model" \
    --test-file dataset/processed/test.jsonl \
    --max-samples 50 \
    --model-name "Base Model Baseline"

print("\n✅ Baseline evaluation completed!")
print("💡 Results saved to JSON file for comparison later")


## Step 6: Train the Model

Run the training script. This will take ~15-30 minutes on a T4 GPU.

⚠️ **IMPORTANT**: Let training complete! Don't interrupt (^C) - you need a fully trained model.


In [ ]:
# Run training
!python train_config_gen_lora.py \
    --mode train \
    --base-model "$BASE_MODEL" \
    --train-file "$TRAIN_FILE" \
    --val-file "$VAL_FILE" \
    --output-dir "$OUTPUT_DIR" \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --learning-rate $LEARNING_RATE \
    --lora-rank $LORA_RANK \
    --lora-alpha $LORA_ALPHA \
    --max-length $MAX_LENGTH

print("\n✅ Training completed!")


## Step 7: Test and Evaluate Trained Model

Test the trained model with sample queries and run full evaluation. Compare results with baseline from Step 5.

⚠️ **IMPORTANT**: Let the evaluation complete! Don't interrupt (^C) - you need the results for comparison.


In [ ]:
# Quick test with sample queries
print("🧪 Quick Test (sample queries)...")
!python train_config_gen_lora.py \
    --mode test \
    --base-model "$BASE_MODEL" \
    --model-path "$OUTPUT_DIR"

print("\n" + "="*70)
print("📊 Full Evaluation (on test set)")
print("="*70)
print("💡 Compare these results with the baseline from Step 5!")
print("⚠️  IMPORTANT: Let this complete to see the results!")
print("")

# Full evaluation on test set
!python train_config_gen_lora.py \
    --mode evaluate \
    --base-model "$BASE_MODEL" \
    --model-path "$OUTPUT_DIR" \
    --test-file dataset/processed/test.jsonl \
    --max-samples 50 \
    --model-name "Trained Model"

print("\n✅ Evaluation completed!")


## Step 8: Compare Baseline vs Trained Model

Compare the baseline metrics with the trained model to see if training improved performance.


In [ ]:
# Compare baseline vs trained model
import json
import os

print("="*70)
print("📈 BASELINE vs TRAINED MODEL - COMPARISON")
print("="*70)
print("")

# Updated file names (without parentheses)
baseline_files = [
    "evaluation_results_base_model_baseline.json",
    "evaluation_results_base_model_(baseline).json",
]
trained_files = [
    "evaluation_results_trained_model.json",
]

# Load results
baseline_results = None
trained_results = None

for bf in baseline_files:
    if os.path.exists(bf):
        with open(bf, 'r') as f:
            baseline_results = json.load(f)
        print(f"✅ Baseline results loaded from: {bf}")
        break

if not baseline_results:
    print(f"⚠️  Baseline results not found!")
    print(f"   Searched for: {', '.join(baseline_files)}")
    print("   Make sure you ran Step 5 (Baseline Evaluation)")

for tf in trained_files:
    if os.path.exists(tf):
        with open(tf, 'r') as f:
            trained_results = json.load(f)
        print(f"✅ Trained model results loaded from: {tf}")
        break

if not trained_results:
    print(f"⚠️  Trained model results not found!")
    print(f"   Searched for: {', '.join(trained_files)}")
    print("   Make sure you ran Step 7 (Full Evaluation)")

print("")

if baseline_results and trained_results:
    print("="*70)
    print("METRICS COMPARISON")
    print("="*70)
    print("")
    
    # YAML Accuracy
    baseline_acc = baseline_results['yaml_accuracy']
    trained_acc = trained_results['yaml_accuracy']
    acc_improvement = trained_acc - baseline_acc
    acc_change = "📈 IMPROVED" if acc_improvement > 0 else "📉 WORSE" if acc_improvement < 0 else "➡️  SAME"
    
    print(f"YAML Syntax Accuracy:")
    print(f"  Baseline:  {baseline_acc:.2f}%")
    print(f"  Trained:   {trained_acc:.2f}%")
    print(f"  Change:    {acc_improvement:+.2f}% {acc_change}")
    print("")
    
    # Error Rate
    baseline_err = baseline_results['error_rate']
    trained_err = trained_results['error_rate']
    err_improvement = baseline_err - trained_err  # Negative is good
    err_change = "📈 IMPROVED" if err_improvement > 0 else "📉 WORSE" if err_improvement < 0 else "➡️  SAME"
    
    print(f"Error Rate:")
    print(f"  Baseline:  {baseline_err:.2f}%")
    print(f"  Trained:   {trained_err:.2f}%")
    print(f"  Change:    {err_improvement:+.2f}% {err_change}")
    print("")
    
    # Success Rate (same as accuracy)
    print(f"Success Rate:")
    print(f"  Baseline:  {baseline_results['success_rate']:.2f}%")
    print(f"  Trained:   {trained_results['success_rate']:.2f}%")
    print(f"  Change:    {acc_improvement:+.2f}% {acc_change}")
    print("")
    
    # Overall assessment
    print("="*70)
    print("OVERALL ASSESSMENT")
    print("="*70)
    
    if acc_improvement > 5:
        print("✅ EXCELLENT: Training significantly improved the model!")
        print(f"   Accuracy improved by {acc_improvement:.2f} percentage points")
    elif acc_improvement > 0:
        print("✅ GOOD: Training improved the model")
        print(f"   Accuracy improved by {acc_improvement:.2f} percentage points")
    elif acc_improvement == 0:
        print("➡️  NEUTRAL: Training did not change performance")
        print("   Consider: more epochs, different learning rate, or more data")
    else:
        print("❌ WORSE: Training decreased performance")
        print(f"   Accuracy decreased by {abs(acc_improvement):.2f} percentage points")
        print("   Consider: lower learning rate, fewer epochs, or check data quality")
    
    print("")
    print("💡 Tips:")
    print("  - If accuracy < 50%: Model may need more training data or epochs")
    print("  - If accuracy improved but still low: Try more epochs or fine-tune hyperparameters")
    print("  - If accuracy decreased: Training may have overfitted, try lower learning rate")
    
elif baseline_results:
    print("⚠️  Only baseline results available. Run Step 7 to get trained model results.")
elif trained_results:
    print("⚠️  Only trained model results available. Run Step 5 to get baseline results.")
else:
    print("❌ No results found. Please run:")
    print("   1. Step 5: Baseline Evaluation")
    print("   2. Step 7: Trained Model Evaluation")


## Step 9: Save Model to Drive

Save the trained model back to Google Drive for later use.


In [ ]:
# Create ZIP of trained model
import zipfile
import os

# Ensure models directory exists
os.makedirs("models", exist_ok=True)

model_zip = f"{OUTPUT_DIR}.zip"
print(f"📦 Creating ZIP: {model_zip}")

# Check if OUTPUT_DIR exists
if not os.path.exists(OUTPUT_DIR):
    print(f"❌ Error: {OUTPUT_DIR} not found!")
    print(f"   Check if training completed successfully")
else:
    with zipfile.ZipFile(model_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(OUTPUT_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, OUTPUT_DIR)
                zipf.write(file_path, arcname)
    
    print(f"✅ Created {model_zip}")
    
    # Copy to Drive
    drive_output = f"{DRIVE_FOLDER}/{os.path.basename(model_zip)}"
    os.makedirs(os.path.dirname(drive_output), exist_ok=True)
    !cp "$model_zip" "$drive_output"
    
    print(f"✅ Model saved to Google Drive: {drive_output}")
    print(f"📊 Model size: {os.path.getsize(model_zip) / 1024 / 1024:.1f} MB")
